In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.4 Hartree–Fock II: The Electron Gas

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.4",
    title="Hartree–Fock II: The Electron Gas",
    blurb="The one many-electron system where exchange can be computed exactly: "
    "jellium. Plane waves diagonalize the Fock operator by symmetry, the exchange "
    "energy comes out in closed form as the density to the one-third power — the "
    "seed of every local-density calculation ever run — and exchange alone "
    "predicts the density of real metals. Then the flaw: a logarithmically "
    "divergent band velocity at the Fermi surface, Hartree–Fock's honest failure, "
    "with screening named as the missing physics and the Ceperley–Alder "
    "correlation energies closing the ledger.",
    difficulty="advanced",
    estimate="120–150 min",
)

## Notebook overview

Movement I closes on the system that anchors all of condensed-matter theory: the
homogeneous electron gas, or *jellium* — electrons in a uniform neutralizing
background, the [§7.9](../07-quantum-statistical-mechanics/fermi-gas-zero-temperature.ipynb)
Fermi gas with the interaction switched back on. Its grace is symmetry: plane waves
remain exact eigenfunctions of the Fock operator, so the Hartree–Fock problem of
[§8.3](hartree-fock-atoms.ipynb), elsewhere a self-consistent grind, here reduces
to *integrals we can do*. What the integrals deliver is the single most consequential
formula in this volume: the exchange energy per particle
$\varepsilon_x = -(3/4)(3/\pi)^{1/3} n^{1/3}$, the density-to-the-one-third law that
[§8.7](kohn-sham.ipynb) will bottle as the local-density approximation and apply to
every atom, molecule, and solid in sight.

The notebook computes four things. The exchange energy, derived by reducing the
k-space double integral and certified by `scipy.integrate.quad` against the closed
form. The Hartree–Fock *band* $\varepsilon(k)$, whose innocent-looking correction
hides a logarithmically divergent slope at $k_F$ — a genuine qualitative failure
(it predicts vanishing density of states at the Fermi level, contradicting every
metal's specific heat from
[§7.10](../07-quantum-statistical-mechanics/fermi-gas-finite-temperature.ipynb)),
whose cure, screening, is named and priced but deferred. The total energy
$E(r_s)$, whose minimum at $r_s = 4.82$ lands squarely among the measured densities
of real alkali metals: cohesion from exchange alone. And the correlation energy,
where theory hands over to the quantum Monte Carlo of Ceperley and Alder
{cite}`ceperley1980` and the Perdew–Zunger fit {cite}`perdewzunger1981` that turned
their handful of numbers into the workhorse functional of a generation. The closing
exercise assembles $\varepsilon_{xc}(r_s)$ — the exact input the Kohn–Sham
construction of [§8.7](kohn-sham.ipynb) is waiting for.

> **Conventions (this notebook).** Hartree atomic units. The gas is unpolarized
> (both spins equally filled). Density is quoted via the Wigner–Seitz radius
> $r_s$ (the radius, in Bohr, of the sphere holding one electron:
> $n = 3/(4\pi r_s^3)$), with $k_F = (9\pi/4)^{1/3}/r_s$ from
> [§7.9](../07-quantum-statistical-mechanics/fermi-gas-zero-temperature.ipynb).
> Integrals use `scipy.integrate.quad` with singular points declared via
> `points=`; energies are per electron.
>
> **How to read the checks.** Each exercise closes with a `validate` call against
> an independent fact: a closed-form integral, a published quantum Monte Carlo
> value, a fitted divergence exponent. A ✓ is strong evidence; a ✗ is a prompt to
> locate the discrepancy, not an automatic verdict.
>
> **Scope.** Exchange exactly; correlation by quotation (the QMC values) and
> parametrization (Perdew–Zunger). The screened effective interaction
> (Thomas–Fermi, Lindhard, RPA) that repairs the Fermi-surface pathology is
> developed in Giustino {cite}`giustino2014`, Ch. 4, and Bruus & Flensberg
> {cite}`bruus2004`, Ch. 13–14; here screening is *named and measured for*, not
> constructed. Martin {cite}`martin2004`, Ch. 5, covers the gas in full.

## Theory in brief

### Why jellium is solvable, and the exchange integral

In a uniform system momentum is a good quantum number, so the determinant of plane
waves filling the Fermi sphere is *self-consistent by symmetry*: the Fock operator
of Eq. {eq}`eq-hf-fock` maps each plane wave to itself. The Hartree term cancels
exactly against the neutralizing background (jellium's founding bargain), leaving
only exchange. Evaluating $K$ between plane waves $\mathbf k$ and $\mathbf k'$
gives the Fourier transform of the Coulomb interaction, $4\pi/|\mathbf k -
\mathbf k'|^2$, and summing over the filled sphere yields the exchange self-energy
of an electron in state $\mathbf k$ (the angular integral is elementary; Martin
{cite}`martin2004`, Ch. 5, or Giustino {cite}`giustino2014`, Ch. 2, carry it out):

```{math}
:label: eq-heg-sigma
\Sigma_x(k) = -\frac{k_F}{\pi}\, F\!\left(\frac{k}{k_F}\right),
\qquad
F(x) = 1 + \frac{1 - x^2}{2x} \ln\left|\frac{1 + x}{1 - x}\right| ,
```

with $F(0) = 2$, $F(1) = 1$. The exchange energy per particle follows by summing
$\tfrac12 \Sigma_x$ over the occupied sphere,

```{math}
:label: eq-heg-ex
\varepsilon_x = \frac{3}{2}\,\frac{1}{k_F^3}\!\int_0^{k_F}\!\! k^2\,\Sigma_x(k)\,dk
= -\frac{3 k_F}{4\pi}
= -\frac{3}{4}\left(\frac{3}{\pi}\right)^{1/3} n^{1/3},
```

where the middle equality hinges on the moment integral
$\int_0^1 x^2 F(x)\,dx = \tfrac12$ — checked numerically below — and the last
substitutes $k_F = (3\pi^2 n)^{1/3}$. This $n^{1/3}$ law is the volume's crown
integral: *the* exchange energy density the local-density approximation will
assign to every point of every inhomogeneous system.

### The pathology at the Fermi surface

The Hartree–Fock band is $\varepsilon(k) = k^2/2 + \Sigma_x(k)$, and
$F$ hides a barb: its derivative diverges logarithmically at $x = 1$, so the band
*velocity* $d\varepsilon/dk$ diverges at $k_F$ and the density of states
$g(\varepsilon_F) \propto 1/|d\varepsilon/dk|$ vanishes. Metals would have zero
electronic specific heat and zero-temperature conductivity anomalies that are
simply not observed — the linear-in-$T$ heat capacity of
[§7.10](../07-quantum-statistical-mechanics/fermi-gas-finite-temperature.ipynb) is
one of the best-verified facts about metals. The diagnosis, due to the 1950s
many-body program: the bare $1/q^2$ Coulomb interaction is too long-ranged, and
the gas itself *screens* it (the $4\pi/q^2$ becomes $4\pi/(q^2 + k_{TF}^2)$ in the
simplest Thomas–Fermi picture), cutting the logarithm off. Screening is
correlation physics — beyond any single determinant — and pricing it is the
correlation energy's job.

### Correlation: where QMC hands theory its numbers

The correlation energy of the gas,
$\varepsilon_c(r_s) = \varepsilon_{\mathrm{exact}} - \varepsilon_{\mathrm{HF}}$, resisted analytic attack for fifty years (the RPA
gets the high-density logarithm; the low-density Wigner crystal the opposite
limit). The decisive numbers came from the stochastic Green's-function Monte Carlo
of Ceperley and Alder (1980) {cite}`ceperley1980`, computed at a handful of $r_s$
values, and Perdew and Zunger {cite}`perdewzunger1981` interpolated them with the
simple form

```{math}
:label: eq-heg-pz
\varepsilon_c^{\mathrm{PZ}}(r_s) =
\begin{cases}
\dfrac{\gamma}{1 + \beta_1\sqrt{r_s} + \beta_2 r_s}, & r_s \ge 1,\\[2ex]
A\ln r_s + B + C\,r_s\ln r_s + D\,r_s, & r_s < 1,
\end{cases}
```

with $\gamma = -0.1423$, $\beta_1 = 1.0529$, $\beta_2 = 0.3334$ and
$A = 0.0311$, $B = -0.048$, $C = 0.0020$, $D = -0.0116$ (Hartree). Coding
Eq. {eq}`eq-heg-pz` and checking it against the published QMC values is not
archaeology: this exact parametrization ships inside essentially every
density-functional code on Earth, and it is the correlation half of the
$\varepsilon_{xc}$ this notebook delivers to [§8.7](kohn-sham.ipynb).

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

# k_F r_s = (9 pi / 4)^(1/3): the Fermi momentum in Wigner-Seitz units
ALPHA_KF = (9.0 * np.pi / 4.0) ** (1.0 / 3.0)


def exchange_factor(x):
    """The exchange band factor F(x) of Eq. eq-heg-sigma.

    F(x) = 1 + (1 - x^2)/(2x) ln|(1+x)/(1-x)|, the angular integral of the
    Coulomb interaction over the Fermi sphere, with the limits F(0) = 2 and
    F(1) = 1 handled explicitly. Its logarithmically divergent slope at x = 1
    is the Hartree-Fock pathology this notebook measures.

    Parameters
    ----------
    x : float
        Momentum in units of the Fermi momentum, k/k_F.

    Returns
    -------
    float
        The dimensionless exchange factor.
    """
    if x < 1e-12:
        return 2.0
    if abs(x - 1.0) < 1e-14:
        return 1.0
    return 1.0 + (1.0 - x * x) / (2.0 * x) * np.log(abs((1.0 + x) / (1.0 - x)))


def eps_c_pz(rs):
    """The Perdew-Zunger parametrization of the correlation energy per electron.

    Eq. eq-heg-pz: the fit through the Ceperley-Alder quantum Monte Carlo
    energies of the unpolarized gas, in Hartree. This exact function ships in
    essentially every LDA implementation; it is the correlation half of the
    eps_xc handed to section 8.7.

    Parameters
    ----------
    rs : float
        Wigner-Seitz radius in Bohr.

    Returns
    -------
    float
        Correlation energy per electron in Hartree.
    """
    if rs >= 1.0:
        return -0.1423 / (1.0 + 1.0529 * np.sqrt(rs) + 0.3334 * rs)
    return 0.0311 * np.log(rs) - 0.048 + 0.0020 * rs * np.log(rs) - 0.0116 * rs

## Exercise 1 — The crown integral: exchange in closed form

Equation {eq}`eq-heg-ex` rests on one moment integral, and the volume's most
consequential formula deserves its certification. The chain is: the band factor
$F$ of Eq. {eq}`eq-heg-sigma` (with its two exact anchors $F(0) = 2$, $F(1) = 1$),
the moment $\int_0^1 x^2 F\,dx = 1/2$, and the assembly
$\varepsilon_x = -3k_F/(4\pi) = -(3/4)(3/\pi)^{1/3} n^{1/3}$.

**Part a)** Check the anchors: evaluate the Setup helper `exchange_factor` at
$x = 10^{-5}$ (a probe of the $x \to 0^+$ limit chosen to dodge the logarithm's
rounding floor; the [§0.1](../00-foundations/floating-point.ipynb) cancellation
story resurfacing inside a many-body integral) and at $x = 1$, against the exact
limits 2 and 1.

**Part b)** Evaluate $\int_0^1 x^2 F(x)\,dx$ with `scipy.integrate.quad`,
declaring the logarithmic point via `points=[1.0]` (and a raised `limit`), against
the exact $1/2$ — the two-line calculation on which the entire local-density
approximation rests.

**Part c)** Assemble $\varepsilon_x$ both ways at $r_s = 2$ and $r_s = 4$: as
$-3k_F/(4\pi)$ with $k_F = (9\pi/4)^{1/3}/r_s$, and as
$-(3/4)(3/\pi)^{1/3}n^{1/3}$ with $n = 3/(4\pi r_s^3)$ (plain `numpy`
arithmetic). The two routes must agree to rounding: the same physics in momentum
and density language, and the second form is the one [§8.7](kohn-sham.ipynb)
bottles.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the LDA's foundation stone

The anchors, the moment, and the two-route agreement at $r_s = 2$
($\varepsilon_x = -0.229083$ Ha) must all hold at quadrature accuracy.

In [ ]:
validate.close(F0, 2.0, "the bottom-of-the-sea limit F(0) = 2", rtol=1e-8)
validate.close(F1, 1.0, "the Fermi-surface value F(1) = 1", rtol=1e-8)
validate.close(moment, 0.5, "the moment integral behind the n^(1/3) law", rtol=1e-9)
validate.close(
    -3.0 * (ALPHA_KF / 2.0) / (4.0 * np.pi), -0.229083, "eps_x at r_s = 2", rtol=1e-5
)
validate.close(
    -3.0 * (ALPHA_KF / 2.0) / (4.0 * np.pi),
    -(3.0 / 4.0)
    * (3.0 / np.pi) ** (1.0 / 3.0)
    * (3.0 / (4.0 * np.pi * 8.0)) ** (1.0 / 3.0),
    "the momentum and density routes agree",
    rtol=1e-12,
)

## Exercise 2 — The Hartree–Fock band

With $\Sigma_x$ in hand, the interacting band $\varepsilon(k) = k^2/2 -
(k_F/\pi)F(k/k_F)$ of Eq. {eq}`eq-heg-sigma` can be drawn against the free
parabola. Exchange *lowers* every state (it is attractive bookkeeping: each
electron avoids its like-spin partners and so feels less repulsion) but lowers
the bottom of the band more than the top ($F(0) = 2$ vs $F(1) = 1$), *widening*
the occupied bandwidth — a real, measurable Hartree–Fock prediction, and
measurably wrong: photoemission bandwidths of simple metals sit much closer to
the free-electron value, another fingerprint of the screening that Hartree–Fock
lacks.

**Part a)** At $r_s = 4$ (a typical metallic density: sodium is $3.93$), tabulate
the occupied bandwidth $\varepsilon(k_F) - \varepsilon(0)$ for the free gas
($k_F^2/2$) and for Hartree–Fock (add $\Sigma_x(k_F) - \Sigma_x(0) = +k_F/\pi$,
assembled from `exchange_factor`), and their ratio.

**Part b)** Plot the two bands over $k \in [0, 1.6\,k_F]$ with the occupied
region shaded and the Fermi points marked. The figure to remember: exchange bends
the band down everywhere, hardest at $k = 0$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2 — the widened band

The exchange contribution to the bandwidth must be exactly $k_F/\pi$ (the
$F(0) - F(1) = 1$ arithmetic), giving a ratio $1 + 2/(\pi k_F) = 2.33$ at
$r_s = 4$.

In [ ]:
validate.close(
    width_hf - width_free, kF_band / np.pi, "the exchange bandwidth widening", rtol=1e-8
)
validate.close(
    width_hf / width_free,
    1.0 + 2.0 / (np.pi * kF_band),
    "the widening ratio at r_s = 4",
    rtol=1e-8,
)

## Exercise 3 — The pathology, measured

Now the barb. The claim in the theory section was that $dF/dx$ diverges like
$\ln|1 - x|$ as $x \to 1$; a claim about an asymptotic law should be a fit. If
the divergence is logarithmic with unit coefficient, then central-difference
slopes of $F$ evaluated ever closer to $x = 1$ should fall on the line
$dF/dx = a\ln(1 - x) + b$ with $a = 1$ — and the band velocity
$d\varepsilon/dk$ then inherits the divergence, driving the density of states at
the Fermi level to zero. That is the pathology: Hartree–Fock predicts every metal
to have a vanishing $g(\varepsilon_F)$, while the measured linear-in-$T$ heat
capacities of
[§7.10](../07-quantum-statistical-mechanics/fermi-gas-finite-temperature.ipynb)
require a healthy finite one.

**Part a)** Compute $dF/dx$ by central differences (`numpy` arithmetic, step
$10^{-7}$) at the twenty points $x = 1 - \delta$ with $\delta$ log-spaced from
$10^{-5}$ to $10^{-2}$ (`numpy.geomspace`), and fit $dF/dx$ against
$\ln(1 - x)$ with `numpy.polyfit` (degree 1). The slope must come out $1.00$:
the logarithm, caught in the act.

**Part b)** Plot the fitted line through the measured slopes, and state the
physical verdict: the divergence is integrable (energies are fine — Exercise 1's
integral converged) but the *velocity* is not, so every Fermi-surface property
(heat capacity, susceptibility, transport) comes out qualitatively wrong.
Screening — the gas rearranging to blunt the $1/q^2$ interaction at long
wavelength — is the missing physics, and it lives beyond the single determinant.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3 — the logarithm, convicted

The fitted coefficient must be $1$ within a percent (the subleading terms at
these $\delta$), and the slopes must grow monotonically as $x \to 1$: an
integrable divergence in the energy, a fatal one in the velocity.

In [ ]:
validate.close(log_coeff, 1.0, "the logarithmic divergence coefficient", rtol=1e-2)
validate.check(
    bool(np.all(np.diff(slopes[::-1]) < 0.0)),
    "the slope grows without bound approaching the Fermi surface",
    f"dF/dx from {slopes[-1]:.2f} at delta = 1e-2 to {slopes[0]:.2f} at 1e-5",
)

## Exercise 4 — Cohesion from exchange alone: the $r_s$ ladder

Assembling kinetic and exchange energies per electron gives the Hartree–Fock
equation of state of the gas,

```{math}
:label: eq-heg-etotal
\frac{E}{N}(r_s) = \frac{3}{5}\frac{k_F^2}{2} - \frac{3k_F}{4\pi}
= \frac{1.1049}{r_s^2} - \frac{0.4582}{r_s} \quad\text{Ha},
```

(the $3/5$ from the kinetic average of
[§7.9](../07-quantum-statistical-mechanics/fermi-gas-zero-temperature.ipynb)).
Kinetic energy loves high density ($r_s^{-2}$), exchange loves low ($r_s^{-1}$
— *negative*), and their competition produces a bound minimum: the gas is
self-cohesive, with no ions doing any work. Where the minimum lands is the
punchline.

**Part a)** Evaluate Eq. {eq}`eq-heg-etotal` on a dense $r_s$ grid, locate the
minimum by `numpy.argmin` refined with a degree-2 `numpy.polyfit` vertex, and
compare $r_s^{\min}$ with the measured valence-electron densities of the alkali
metals (Na $3.93$, K $4.86$, Rb $5.20$, Cs $5.62$; Ashcroft–Mermin Table 1.1).

**Part b)** Plot $E(r_s)$ with its kinetic and exchange parts separated and the
alkali densities marked. Exchange alone parks the minimum at $r_s = 4.82$, in
the middle of the alkali row: the first ab-initio prediction of how dense a
metal wants to be.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4 — the alkali-density prediction

The minimum must sit at $r_s = 4.823$ with $E = -0.0475$ Ha, inside the measured
alkali window $[3.93, 5.62]$.

In [ ]:
validate.close(
    rs_min, 4.823, "the Hartree-Fock equilibrium density of jellium", rtol=1e-3
)
validate.close(E_min, -0.0475, "the cohesive energy per electron", rtol=1e-2)
validate.check(
    ALKALIS["Na"] < rs_min < ALKALIS["Cs"],
    "the prediction lands inside the measured alkali row",
    f"3.93 < {rs_min:.2f} < 5.62",
)

## Exercise 5 — Correlation: the Ceperley–Alder handoff

What Hartree–Fock leaves out, quantum Monte Carlo measured. The published
Ceperley–Alder energies for the unpolarized gas {cite}`ceperley1980` give
correlation energies per electron of $-0.0451$, $-0.0282$, $-0.0186$, and
$-0.0115$ Ha at $r_s = 2, 5, 10, 20$ (their Rydberg values halved), and the
Perdew–Zunger form of Eq. {eq}`eq-heg-pz` interpolates them.

**Part a)** Code Eq. {eq}`eq-heg-pz` (the Setup helper `eps_c_pz`) and evaluate
it at the four published $r_s$ values; report the relative deviations (they
must sit within the fit's advertised percent-level accuracy).

**Part b)** Plot the parametrization through the QMC points across
$r_s \in [0.5, 25]$ (log $r_s$ axis), with the $r_s < 1$ high-density branch
(the RPA logarithm) distinguished. At metallic density correlation is roughly a
quarter of exchange — smaller, but the same order, and never negligible.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5 — the fit meets its sources

The parametrization must reproduce all four published values within $1\%$, and
the two branches of Eq. {eq}`eq-heg-pz` must join continuously at $r_s = 1$.

In [ ]:
validate.check(
    all(d < 0.01 for d in devs.values()),
    "Perdew-Zunger reproduces the Ceperley-Alder values within 1%",
    "max deviation " + f"{100 * max(devs.values()):.2f}%",
)
validate.close(
    eps_c_pz(1.0 - 1e-9),
    eps_c_pz(1.0),
    "the two branches join at r_s = 1",
    rtol=0.0,
    atol=2e-4,
)

## Exercise 6 — The deliverable: $\varepsilon_{xc}(r_s)$

The movement's closing act is a handoff. The exchange of Exercise 1 and the
correlation of Exercise 5 combine into the exchange-correlation energy per
electron of the uniform gas,
$\varepsilon_{xc}(r_s) = \varepsilon_x(r_s) + \varepsilon_c(r_s)$ — the *only*
input the local-density approximation needs. When
[§8.7](kohn-sham.ipynb) writes
$E_{xc}^{\mathrm{LDA}}[n] = \int n(\mathbf r)\,\varepsilon_{xc}\big(n(\mathbf r)\big)\,d\mathbf r$,
every value it looks up comes off the curve assembled here.

**Part a)** Tabulate $\varepsilon_x$, $\varepsilon_c$, $\varepsilon_{xc}$, and
the ratio $\varepsilon_c/\varepsilon_x$ at $r_s = 1, 2, 4, 6, 10$ (`numpy`
arithmetic on the two closed forms). The ratio grows with $r_s$: dilution makes
the gas *more* correlated, the road to the Wigner crystal (the $r_s \to \infty$
limit where electrons localize on a lattice, $\varepsilon \to -0.896/r_s$;
named here, computed nowhere — it has never been reached by experiment in
three dimensions).

**Part b)** Plot the three curves on one axis across $r_s \in [1, 12]$. This
figure *is* the LDA: everything the workhorse of materials science knows about
electron interaction, on one pair of axes.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6 — the handoff, checked

At $r_s = 4$: $\varepsilon_x = -0.11454$ Ha, correlation $28\%$ of it, and the
ratio must grow monotonically across the table (dilute = correlated).

In [ ]:
validate.close(
    -3.0 * (ALPHA_KF / 4.0) / (4.0 * np.pi), -0.114541, "eps_x at r_s = 4", rtol=1e-5
)
validate.close(
    ratios[4.0], 0.280, "correlation-to-exchange ratio at metallic density", rtol=1e-2
)
validate.check(
    bool(np.all(np.diff([ratios[v] for v in (1.0, 2.0, 4.0, 6.0, 10.0)]) > 0)),
    "dilution makes the gas more correlated",
    f"c/x from {ratios[1.0]:.3f} at r_s = 1 to {ratios[10.0]:.3f} at r_s = 10",
)

:::{admonition} With your assistant
:class: tip
The spin-polarized gas repeats this notebook with the two spin channels filled
differently. Have your assistant derive and code the fully polarized exchange
energy (all electrons one spin: $\varepsilon_x^{P} = 2^{1/3}\varepsilon_x^{U}$
at equal *total* density), then run the check that is yours alone: at $r_s = 4$
the polarized exchange must equal $2^{1/3} \times (-0.11454)$ Ha
(`numpy.isclose`, `rtol=1e-10`), and the polarized *kinetic* energy must be
$2^{2/3}$ times the unpolarized one — so polarization costs kinetic energy
faster than exchange repays it at metallic density. The check is yours.
:::

## Notebook summary

Movement I closes with the many-electron problem solved exactly in the one place
symmetry allows. The exchange band factor was certified at its anchors
($F(0) = 2$, $F(1) = 1$), its moment integral $\int x^2F = 1/2$ evaluated to
$10^{-12}$, and the crown formula assembled two ways:
$\varepsilon_x = -3k_F/4\pi = -(3/4)(3/\pi)^{1/3}n^{1/3} = -0.458/r_s$ Ha. The
Hartree–Fock band widened the occupied states by $1 + 2/\pi k_F$ (a factor $2.33$ at
$r_s = 4$) and hid the pathology: the band velocity's logarithmic divergence at
$k_F$ was convicted by fit (coefficient $1.00$), with vanishing Fermi-level
density of states as the physical casualty and screening as the named cure. The
equation of state $1.105/r_s^2 - 0.458/r_s$ put its minimum at $r_s = 4.82$,
inside the measured alkali-metal row. Correlation arrived by handoff — the four
Ceperley–Alder values reproduced by the Perdew–Zunger form within $0.7\%$ — and
the movement's deliverable, $\varepsilon_{xc}(r_s)$, is now a curve this course
computed and certified end to end.

## Outlook

- The $n^{1/3}$ exchange law and the PZ correlation together are the *entire*
  physics input of the local-density approximation. [§8.7](kohn-sham.ipynb)
  applies them, point by point, to atoms whose density is anything but uniform —
  and the surprise will be how well a uniform-gas fact travels.
- Screening, named here as the cure for the Fermi-surface logarithm, returns
  twice: as the Thomas–Fermi wavevector inside [§8.5](thomas-fermi.ipynb)'s
  statistical picture, and as the RPA screened interaction $W$ at the heart of
  the GW method in [§8.14](quasiparticles-gw.ipynb).
- The Wigner crystal named in Exercise 6 is the $r_s \to \infty$ endpoint of the
  correlation road; the Mott transition of [§8.13](hubbard-model.ipynb) is its
  lattice cousin, and there the volume will finally *compute* a
  correlation-driven localization.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()